In [1]:
import numpy as np
import pandas as pd
import zipfile

In [2]:
with zipfile.ZipFile("abalone.zip", 'r') as zip_ref:
    zip_ref.extractall("abalone_data")

In [3]:
column_names = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight", "Rings"
]

data = pd.read_csv("abalone_data/abalone.data", 
                   header=None, 
                   names=column_names)

# Print basic info
print("Number of rows:", len(data))
print("Column names:", data.columns.tolist())
print("\nFirst 5 rows:")
print(data.head())

# Checkpoint:
# input: physical measurements of abalone
# output: Rings (age indicator)
# why numeric: Rings is count of growth rings (integer)

Number of rows: 4177
Column names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']

First 5 rows:
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


In [4]:
# Target transformation
y = (data["Rings"] + 1.5).values.reshape(-1, 1)

# y represents estimated age (numeric regression target)

In [5]:
# Select exactly 3 numeric features

features = ["Length", "Diameter", "Whole_weight"]

X = data[features].values

# Justification:
# Length: overall size indicator
# Diameter: width contributes to maturity
# Whole_weight: heavier abalone likely older

In [6]:
# 80/20 split
N = len(X)
indices = np.random.permutation(N)

split = int(0.8 * N)

train_idx = indices[:split]
test_idx = indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3341, 3)
Test shape: (836, 3)


In [7]:
# Compute mean and std from training only
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Normalize
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Checkpoint:
# Normalization prevents large-scale features
# from dominating gradient updates.

In [8]:
def forward(X, w, b):
    """
    Computes y_hat = Xw + b
    """
    y_hat = X @ w + b
    
    print("Shape X:", X.shape)
    print("Shape w:", w.shape)
    print("Shape b:", np.array(b).shape)
    print("Shape y_hat:", y_hat.shape)
    
    # Checkpoint:
    # parameters are w and b
    # number of parameters = d weights + 1 bias
    
    return y_hat

In [9]:
def mse(y, y_hat):
    loss = np.mean((y - y_hat) ** 2)
    
    # Checkpoint:
    # why square: penalizes large errors more
    # large mistakes become very expensive
    
    return loss

In [10]:
def grad_w(X, y, y_hat):
    N = len(y)
    dW = (2/N) * X.T @ (y_hat - y)
    return dW

def grad_b(y, y_hat):
    N = len(y)
    db = (2/N) * np.sum(y_hat - y)
    return db

# Checkpoint:
# gradient = direction of steepest increase
# subtracting gradient reduces loss

# meaning of large gradient:
# model is far from minimum

# effect of too-large learning rate:
# may overshoot and diverge

In [11]:
# Initialize parameters
d = X_train.shape[1]
w = np.random.randn(d, 1) * 0.01
b = 0.0

learning_rate = 0.01
epochs = 200

# Initial expectation:
# Loss should decrease gradually, not instantly.

for epoch in range(epochs):
    
    # 1) Forward
    y_hat = X_train @ w + b
    
    # 2) Loss
    loss = mse(y_train, y_hat)
    
    # 3) Gradients
    dW = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)
    
    # 4) Update
    w -= learning_rate * dW
    b -= learning_rate * db
    
    if epoch % 20 == 0:
        print("Epoch:", epoch, "Loss:", loss)

# Revised expectation:
# Loss decreases smoothly if learning rate is stable.

Epoch: 0 Loss: 140.9359417607174
Epoch: 20 Loss: 65.54276814223462
Epoch: 40 Loss: 32.99248030028581
Epoch: 60 Loss: 18.580704991610364
Epoch: 80 Loss: 12.165042049939366
Epoch: 100 Loss: 9.305153693924304
Epoch: 120 Loss: 8.029371913563518
Epoch: 140 Loss: 7.459602913935439
Epoch: 160 Loss: 7.204540056301037
Epoch: 180 Loss: 7.08978096483362


In [12]:
# Test predictions
y_test_hat = X_test @ w + b

# Test MSE
test_mse = mse(y_test, y_test_hat)

# Test MAE
test_mae = np.mean(np.abs(y_test - y_test_hat))

print("Test MSE:", test_mse)
print("Test MAE:", test_mae)

Test MSE: 7.152487861035528
Test MAE: 1.9200116946250607


In [13]:
print("\nSample Predictions:\n")

for i in range(5):
    true_age = y_test[i][0]
    pred_age = y_test_hat[i][0]
    abs_error = abs(true_age - pred_age)
    
    print("True:", true_age,
          "Predicted:", round(pred_age,2),
          "Abs Error:", round(abs_error,2))

# Checkpoint:
# systematic errors: older abalones often slightly underpredicted
# observed bias: model tends to regress toward mean age


Sample Predictions:

True: 11.5 Predicted: 11.85 Abs Error: 0.35
True: 10.5 Predicted: 11.67 Abs Error: 1.17
True: 5.5 Predicted: 6.22 Abs Error: 0.72
True: 9.5 Predicted: 11.18 Abs Error: 1.68
True: 8.5 Predicted: 10.21 Abs Error: 1.71
